# 🌊 04 · Diffraction, Resolution, and the Point-Spread Function
### Why light itself sets a hard limit on what any lens can show you

**Part B · Seeing like a microscope** · Module 04 of 37 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/04_diffraction_resolution_and_the_point_spread_function.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, Matplotlib, SciPy |
| **Prerequisites** | Module 03 |
| **Estimated time** | 75–90 minutes |
| **Instrumentation topics covered** | Huygens-Fresnel diffraction; the point-spread function (PSF) as a Fourier-optics calculation; the Airy pattern and a derivation (not just a quoted formula) of the Rayleigh resolution criterion; the optical transfer function (OTF) and spatial-frequency cutoff; every image as an unavoidable convolution with the PSF |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> Every PSF, Airy pattern and OTF in this notebook is computed by Fourier optics from a simulated circular aperture; nothing is read from a camera or a file.

## 🎯 Learning objectives

1. Explain, in terms of wave interference, why even a perfect lens cannot focus light to an
   infinitely small point — and why this has nothing to do with manufacturing imperfections.
2. Compute a diffraction-limited point-spread function (PSF) as the squared magnitude of the Fourier
   transform of a lens's pupil function, and relate its width to numerical aperture and wavelength.
3. Derive the Rayleigh two-point resolution criterion from the Airy pattern's first zero, directly
   from a simulated PSF, rather than taking the formula on faith.
4. Explain the point-spread function as the fundamental "blur kernel" of an imaging system, and the
   optical transfer function (OTF) as the corresponding hard cutoff on spatial frequency.
5. Explain why every image used since Module 02 — and every image any real microscope ever
   produces — is already a convolution with this PSF, before any noise or downstream processing.

## 🗺️ Contents

1. Huygens-Fresnel diffraction: why a point source doesn't stay a point
2. The point-spread function as a Fourier-optics calculation
3. The Airy pattern and the Rayleigh resolution criterion, derived from simulation
4. The optical transfer function: resolution in frequency space
5. Living with the PSF: every image is already a convolution

## 🧫 The biological question

Two vesicles really are 180 nanometers apart inside a living cell. Does the microscope image show
one blob or two? The honest answer does not depend on the camera's pixel size, the brightness of the
fluorophores, or how clever a downstream image-processing algorithm might be — it depends entirely on
whether 180 nanometers is above or below a hard physical limit set by the wave nature of light
itself, diffracting through the objective's finite aperture. Module 03 showed that numerical aperture
sets *a* resolution limit; this module derives *why*, from first principles, and shows exactly what
that limit looks like baked into every single image a microscope will ever produce — a limit no
amount of computation downstream can see past, only estimate around.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
print('No GPU is needed anywhere in this notebook — everything is plain NumPy/SciPy/Matplotlib.')

In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import j1
SEED = 4
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11

### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.1; copied so this notebook runs alone) ──
import numpy as np

def simulate_psf(grid: int = 160, na: float = 0.9, wavelength_um: float = 0.55,
                  pixel_um: float = 0.035, aberrations: dict[tuple[int, int], float] | None = None) -> np.ndarray:
    """A diffraction-limited (or aberrated) point-spread function by Fourier optics: the
    incoherent PSF is the squared magnitude of the Fourier transform of the pupil function
    (Fraunhofer diffraction of a circular aperture of the given NA). ``aberrations`` optionally
    maps (n, m) Zernike orders to a wavefront-error coefficient in WAVES, added as phase across
    the pupil before transforming (e.g. ``{(4, 0): 0.25}`` for 0.25 waves of primary spherical
    aberration). With no aberrations this reproduces the classic Airy pattern.
    """
    freq = np.fft.fftshift(np.fft.fftfreq(grid, d=pixel_um))
    fx, fy = np.meshgrid(freq, freq)
    rho = np.hypot(fx, fy) / (na / wavelength_um)
    theta = np.arctan2(fy, fx)
    pupil = (rho <= 1.0).astype(complex)
    if aberrations:
        rho_clipped = np.clip(rho, 0, 1)
        phase = np.zeros_like(rho)
        for (n, m), coeff in aberrations.items():
            phase = phase + coeff * zernike(n, m, rho_clipped, theta)
        pupil = pupil * np.exp(2j * np.pi * phase)
    field = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(pupil)))
    psf = np.abs(field) ** 2
    return psf / psf.sum()


def rayleigh_resolution(wavelength_um: float, na: float) -> float:
    """The Rayleigh resolution criterion (um): the minimum resolvable centre-to-centre distance
    between two incoherent point sources, 0.61 * wavelength / NA."""
    return 0.61 * wavelength_um / na


def abbe_resolution(wavelength_um: float, na: float) -> float:
    """Abbe's diffraction-limited resolution (um) for a periodic structure in coherent
    transmitted light: wavelength / (2 NA)."""
    return wavelength_um / (2.0 * na)

---
## 1 · Huygens-Fresnel diffraction: why a point source doesn't stay a point

Huygens' principle treats every point on a propagating wavefront as a source of its own tiny
spherical "wavelet"; the wavefront a moment later is the interference pattern of all those wavelets
added together. Pass a wave through any finite opening — a slit, or a lens's circular rim — and the
wavelets launched from different parts of that opening travel slightly different distances to any
point downstream, so they arrive slightly out of phase and interfere. The result is **diffraction**: a
beam that was perfectly parallel before the opening necessarily spreads out after it, by an amount
that gets *larger* as the opening gets *smaller*. A lens can refocus light, but it cannot undo this
spreading — the lens itself is a finite opening, so it diffracts the light it collects by exactly the
same physics.

A one-dimensional slit is the simplest case to make this concrete, before Section 2 tackles the full
two-dimensional circular aperture of a real lens: a slit of width $a$ illuminated by light of
wavelength $\lambda$ produces a far-field diffraction pattern $I(\theta) \propto
\text{sinc}^2(\pi a \sin\theta / \lambda)$, whose central peak gets wider precisely as $a$ shrinks.

In [ ]:
# ── 1.1 single-slit diffraction: narrower aperture -> wider spread, by physics alone ──
wavelength_um = 0.55
theta = np.linspace(-0.3, 0.3, 2000)  # radians

fig, ax = plt.subplots(figsize=(7, 3.3))
for slit_width_um in [2.0, 6.0, 20.0]:
    x = np.pi * slit_width_um * np.sin(theta) / wavelength_um
    x = np.where(np.abs(x) < 1e-9, 1e-9, x)
    intensity = (np.sin(x) / x) ** 2
    ax.plot(np.degrees(theta), intensity, label=f'slit width = {slit_width_um:.0f} um')
ax.set_xlabel('diffraction angle (degrees)'); ax.set_ylabel('relative intensity')
ax.set_title('Single-slit diffraction: a smaller aperture spreads light MORE, not less')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The narrowest slit (2 micrometers) produces the *widest* central
diffraction peak; the widest slit (20 micrometers) produces the *narrowest* one. This inverse
relationship between aperture size and spread angle is the entire reason a microscope objective's
numerical aperture — which sets the angular size of the "opening" collecting light — controls
resolution. Section 2 repeats this exact calculation for a two-dimensional circular aperture, which
is what a real lens actually is.

✏️ **Your turn.** Add a fourth curve for `slit_width_um = 0.5`. At what point does treating this
as "far-field, paraxial diffraction" start to become a stretch? (You do not need a precise answer —
just notice how extreme the spread angle becomes.)

---
## 2 · The point-spread function as a Fourier-optics calculation

A microscope objective's circular rim is, in diffraction terms, just a two-dimensional
aperture — a **pupil function** that is 1 inside a disk of a radius set by the numerical aperture and
0 outside it. A fundamental result of Fourier optics is that the image of a single, infinitely small
point source — the system's **point-spread function (PSF)** — is simply the squared magnitude of the
two-dimensional Fourier transform of that pupil function:

$$\text{PSF}(x, y) = \left| \mathcal{F}\{\text{pupil}(f_x, f_y)\} \right|^2$$

This is exactly what `simulate_psf` computes: it builds a disk-shaped pupil of radius $NA/\lambda$ in
spatial-frequency units, takes its 2-D FFT, and squares the magnitude. (Module 12's frequency-domain
ideas are the same FFT machinery pointed at a different physical quantity — here the aperture itself,
rather than a captured image.)

In [ ]:
# ── 2.1 compute and display a diffraction-limited PSF ──
na = 0.9
wavelength_um = 0.55
pixel_um = 0.035
grid = 160
psf = simulate_psf(grid=grid, na=na, wavelength_um=wavelength_um, pixel_um=pixel_um)
extent_um = grid * pixel_um / 2

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
im = axes[0].imshow(psf, cmap='inferno', extent=[-extent_um, extent_um, -extent_um, extent_um])
axes[0].set_xlim(-1.5, 1.5); axes[0].set_ylim(-1.5, 1.5)
axes[0].set_xlabel('x (um)'); axes[0].set_ylabel('y (um)')
axes[0].set_title(f'Simulated PSF (NA={na}, lambda={wavelength_um} um)')
plt.colorbar(im, ax=axes[0], fraction=0.046)

center = grid // 2
radial_profile = psf[center, center:] / psf[center, center]
radius_um = np.arange(len(radial_profile)) * pixel_um
axes[1].plot(radius_um, radial_profile, color='teal')
axes[1].set_xlim(0, 1.5)
axes[1].set_xlabel('radius (um)'); axes[1].set_ylabel('normalized intensity')
axes[1].set_title('Radial cross-section through the PSF centre')
plt.tight_layout(); plt.show()

🔍 **What just happened?** Even though the pupil function is a perfectly sharp-edged disk (a
"perfect lens" with no aberrations at all), the resulting PSF is not a point — it is a bright central
peak surrounded by faint concentric rings. This is the famous **Airy pattern**, and it exists purely
because of the pupil's finite size, exactly the single-slit spreading from Section 1 but now in two
dimensions and with circular symmetry.

✏️ **Your turn.** Change `na` to `0.3` and re-run. How does the central peak's width change?
Which direction does that match from Section 1's slit-width experiment, and does it match the
inverse-NA scaling Module 03's `rayleigh_resolution` function implies?

---
## 3 · The Airy pattern and the Rayleigh resolution criterion, derived from simulation

The Airy pattern has a closed-form radial profile, derived from exactly the same Fourier
integral `simulate_psf` evaluates numerically:

$$I(r) \propto \left[\frac{2 J_1(x)}{x}\right]^2, \qquad x = \frac{2\pi \, NA}{\lambda} r$$

where $J_1$ is the Bessel function of the first kind, order 1. This profile's first zero — the edge
of the bright central disk — occurs at $x = 3.8317$, which converts to a physical radius of
$r_{Airy} = 0.61\,\lambda / NA$. **This is exactly the Rayleigh resolution formula** — not a separate
empirical rule, but a direct geometric fact about where the Airy pattern's first dark ring falls.

The classic **Rayleigh criterion** then defines two equally-bright point sources as "just resolved"
when they are separated by exactly this radius: when the peak of one Airy pattern sits on top of the
first dark ring of the other.

In [ ]:
# ── 3.1 cross-check the simulated PSF's radial profile against the analytic Airy formula ──
x = 2 * np.pi * na / wavelength_um * radius_um
x_safe = np.where(np.abs(x) < 1e-9, 1e-9, x)
airy_analytic = (2 * j1(x_safe) / x_safe) ** 2
airy_analytic[0] = 1.0  # the x -> 0 limit

airy_first_zero_radius_um = 0.61 * wavelength_um / na  # == rayleigh_resolution(...)
predicted = rayleigh_resolution(wavelength_um, na)
print(f'Rayleigh radius from the closed-form Airy zero:  {airy_first_zero_radius_um:.4f} um')
print(f'Rayleigh radius from course_toolkit.rayleigh_resolution: {predicted:.4f} um  (should match)')

fig, ax = plt.subplots(figsize=(7, 3.3))
ax.plot(radius_um, radial_profile, color='teal', lw=2, label='simulated PSF (FFT of pupil)')
ax.plot(radius_um, airy_analytic, color='crimson', ls='--', lw=1.5, label='analytic Airy formula')
ax.axvline(airy_first_zero_radius_um, color='gray', ls=':', label=f'Rayleigh radius = {airy_first_zero_radius_um:.3f} um')
ax.set_xlim(0, 1.2)
ax.set_xlabel('radius (um)'); ax.set_ylabel('normalized intensity')
ax.set_title('Simulated PSF matches the closed-form Airy pattern')
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

🧪 **Ground-truth check.** The FFT-simulated PSF and the independent, closed-form Bessel-function
formula overlap almost exactly, and both go to zero at the same radius that `rayleigh_resolution`
predicts from the simple $0.61\lambda/NA$ formula — three independent routes to the same number,
because they are three descriptions of the same underlying physics.

In [ ]:
# ── 3.2 the two-point Rayleigh criterion, demonstrated directly on simulated PSFs ──
def shifted_psf(dx_px, dy_px=0):
    return np.roll(np.roll(psf, dx_px, axis=1), dy_px, axis=0)

rayleigh_px = int(round(airy_first_zero_radius_um / pixel_um))
separations_px = [max(1, rayleigh_px // 2), rayleigh_px, int(rayleigh_px * 1.6)]

fig, axes = plt.subplots(1, len(separations_px), figsize=(12, 3.2))
for ax, sep_px in zip(axes, separations_px):
    combined = shifted_psf(-sep_px // 2) + shifted_psf(sep_px // 2)
    row = combined[center, center - 3 * rayleigh_px: center + 3 * rayleigh_px]
    x_um = (np.arange(len(row)) - len(row) / 2) * pixel_um
    ax.plot(x_um, row / row.max(), color='teal')
    sep_um = sep_px * pixel_um
    dip = row.min() / row.max() if sep_px < len(row) else np.nan
    ax.set_title(f'separation = {sep_um:.3f} um\n({sep_um / airy_first_zero_radius_um:.2f} x Rayleigh)',
                 fontsize=9)
    ax.set_xlabel('x (um)')
plt.suptitle('Two point sources at increasing separation: when do they become "two" rather than "one"?')
plt.tight_layout(); plt.show()

🔍 **What just happened?** At well under the Rayleigh separation, the two PSFs merge into one
single peak with no dip at all — two physically distinct points produce an image indistinguishable
from one point. Right around the Rayleigh separation, a shallow-but-genuine dip appears between the
two peaks — this is the conventional threshold for calling two points "resolved". Beyond it, the dip
deepens further and the two peaks are unambiguous. Nothing about pixel size or camera electronics
appears anywhere in this calculation — the limit is set entirely by the PSF.

### 📘 Lecture notes — the Rayleigh criterion is a statement about the PSF, not the camera

**Definition.** The Rayleigh resolution criterion, $0.61\lambda/NA$, is the radius of the first dark
ring of the diffraction-limited PSF — not an arbitrary convention, but a direct geometric feature of
the Airy pattern derived in this section.

**Worked example.** Oversampling an image with a smaller camera pixel size can make a blurred pair of
points easier to *see* as two peaks on a plot, but it can never move the Rayleigh radius itself —
sampling more finely records the same already-overlapped PSFs in more detail, it does not un-overlap
them.

**One memorable point.** *Diffraction sets how much true information about the specimen the PSF
physically lets through; pixel size only decides how faithfully that (already limited) information
gets recorded — a camera with twice the pixels does not have twice the resolution.*

---
## 4 · The optical transfer function: resolution in frequency space

Section 3 looked at resolution in real (spatial) coordinates — how close two points can be. The
same limit has an equally important frequency-space description. The **optical transfer function
(OTF)** is the normalized Fourier transform of the PSF; its magnitude, the **modulation transfer
function (MTF)**, says how faithfully the system transmits each spatial frequency present in the
specimen, from $1.0$ (perfect transmission) down to $0$ (frequency completely lost). For an
aberration-free circular aperture, the MTF falls to exactly zero at a hard cutoff spatial frequency

$$f_{cutoff} = \frac{2\,NA}{\lambda}$$

and stays at zero beyond it. This is the frequency-domain twin of the Rayleigh radius: any specimen
structure finer than $1/f_{cutoff}$ is not merely blurred, it is **completely absent from the data**,
exactly like the "one-way door" idea from Module 03 but now expressed per spatial frequency.

In [ ]:
# ── 4.1 compute the OTF/MTF from the PSF, and locate the hard frequency cutoff ──
otf = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(psf)))
mtf = np.abs(otf)
mtf = mtf / mtf.max()
freq_axis = np.fft.fftshift(np.fft.fftfreq(grid, d=pixel_um))  # cycles / um

cutoff_theory = 2 * na / wavelength_um
mtf_row = mtf[center, center:]
freq_row = freq_axis[center:]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
im = axes[0].imshow(mtf, cmap='viridis', extent=[freq_axis[0], freq_axis[-1], freq_axis[0], freq_axis[-1]])
axes[0].set_xlim(-6, 6); axes[0].set_ylim(-6, 6)
axes[0].set_xlabel('spatial frequency (cycles/um)'); axes[0].set_ylabel('spatial frequency (cycles/um)')
axes[0].set_title('2-D MTF')
plt.colorbar(im, ax=axes[0], fraction=0.046)

axes[1].plot(freq_row, mtf_row, color='darkorange')
axes[1].axvline(cutoff_theory, color='gray', ls='--', label=f'2 NA/lambda = {cutoff_theory:.2f} cyc/um')
axes[1].set_xlim(0, 6)
axes[1].set_xlabel('spatial frequency (cycles/um)'); axes[1].set_ylabel('MTF')
axes[1].set_title('Radial MTF profile')
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The MTF profile falls smoothly from 1.0 at zero frequency down to
essentially zero right around the predicted cutoff $2NA/\lambda$ — frequencies beyond that line carry
no information through this optical system at all, regardless of how bright the specimen structure at
that frequency actually is. A later module on denoising and restoration can only ever work with
frequencies the MTF has *not* already zeroed out; beyond the cutoff there is nothing left to
restore.

✏️ **Your turn.** Halve `na` to `0.45`, recompute `otf`/`mtf`, and compare the new cutoff frequency
to the old one. Does it match the `2 NA / lambda` formula's prediction?

### 📘 Lecture notes — the OTF is where "resolution" and "contrast" meet

**Definition.** The MTF gives, for every spatial frequency in the specimen, the fraction of its
original contrast that survives into the image — not an all-or-nothing resolved/unresolved question,
but a smooth, frequency-by-frequency attenuation that only reaches exactly zero at the cutoff.

**Worked example.** A specimen structure at half the cutoff frequency is not "fully resolved" just
because it is below the cutoff — the MTF there is typically well under 1.0, so its contrast in the
final image is already visibly reduced, even though the structure is, strictly, resolvable.

**One memorable point.** *"Resolved" is not binary — it is the point where the MTF's gently falling
curve crosses whatever contrast the specimen and the detector's noise floor can still make visible.*

---
## 5 · Living with the PSF: every image is already a convolution

Every point in a real specimen produces its own copy of the PSF in the image, scaled by that
point's brightness and added to all the others. Mathematically, the image a microscope forms is
exactly the convolution of the true specimen with the system's PSF:

$$\text{image} = \text{specimen} * \text{PSF}$$

This is true before a single photon is counted by the camera, before any read noise or quantization —
it is a property of the optics alone. Every image used anywhere else in this course, synthetic or
otherwise, is implicitly standing in for an already-PSF-convolved image; this section makes that
convolution explicit, once, so it is never invisible again.

In [ ]:
# ── 5.1 convolve a code-generated toy specimen with the simulated PSF ──
size = grid
yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing='ij')
specimen = np.zeros((size, size))
centers = rng.uniform(30, size - 30, size=(14, 2))
for cy, cx in centers:
    radius = rng.uniform(1.5, 4.0)
    specimen += np.exp(-(((yy - cy) ** 2 + (xx - cx) ** 2) / (2 * radius ** 2)))
specimen = np.clip(specimen, 0, 1)

image = np.real(np.fft.ifft2(np.fft.fft2(specimen) * np.fft.fft2(np.fft.ifftshift(psf))))
image = np.clip(image, 0, None)

fig, axes = plt.subplots(1, 2, figsize=(9, 4.2))
axes[0].imshow(specimen, cmap='gray'); axes[0].set_title('Ground-truth specimen\n(never directly observable)')
axes[1].imshow(image / image.max(), cmap='gray'); axes[1].set_title('What the microscope actually records\n(specimen (*) PSF)')
for ax in axes:
    ax.axis('off')
plt.tight_layout(); plt.show()

🔍 **What just happened?** Point-like features that are far apart in the ground-truth specimen
remain distinct after convolution; features placed closer together than roughly the Rayleigh radius
from Section 3 visibly merge into single blobs in the recorded image — exactly the same physics as
Section 3's two-point demonstration, now shown on a realistic, many-point specimen rather than just
two isolated sources.

✏️ **Your turn.** Reduce `na` to `0.4`, recompute `psf`, and re-run the convolution. Which of the 14
simulated points merge together that were previously distinct? This is exactly the trade-off a
microscopist accepts whenever they choose a lower-NA, longer-working-distance objective, as in
Module 03.

## 📌 Key takeaways

- Diffraction through any finite aperture is unavoidable; a smaller aperture always spreads light
  *more*, never less.
- A microscope's point-spread function is the squared magnitude of the Fourier transform of its
  pupil function — the Airy pattern is this calculation's result for a plain circular aperture.
- The Rayleigh resolution criterion is not a separate empirical rule — it is exactly the radius of
  the Airy pattern's first dark ring, derivable directly from the PSF.
- The optical transfer function expresses the same limit in frequency space: spatial frequencies
  beyond $2NA/\lambda$ are not attenuated, they are completely absent from the data.
- Every image a microscope ever produces is already a convolution of the true specimen with the PSF,
  before any noise or downstream processing is applied.

## ✅ Checkpoint

**1.** Doubling a camera's pixel density (smaller pixels, same field of view) is proposed as a way to
"increase resolution." Evaluate this claim using what this module derived.

<details><summary>Answer</summary>
It does not increase resolution in the diffraction sense. The Rayleigh radius and the OTF cutoff
depend only on wavelength and NA, properties of the optics, not the camera. Smaller pixels can better
*sample* an already-blurred image (useful up to the Nyquist rate implied by the optical cutoff, and
necessary to avoid aliasing), but they cannot recover spatial frequencies the optics never
transmitted in the first place.
</details>

**2.** Two points are separated by exactly twice the Rayleigh radius. Does the OTF/MTF framework
predict their frequency content is "cut off"? Why or why not?

<details><summary>Answer</summary>
No — the Rayleigh radius describes a *specific* two-point separation threshold, not a frequency
cutoff. A real-space separation of twice the Rayleigh radius corresponds to a (lower) spatial
frequency that is comfortably below the $2NA/\lambda$ cutoff, so its contrast is attenuated by the MTF
but not zeroed; the two points are easily resolved, consistent with Section 3's two-point simulation.
</details>

**3.** Why does this module describe the image as "specimen convolved with PSF" rather than
"specimen plus some blur added afterward"?

<details><summary>Answer</summary>
Convolution is the correct mathematical operation because every specimen point independently produces
its own scaled, shifted copy of the PSF, and the recorded image is the sum of all of those copies.
This is fundamentally different from adding a separate "blur layer" afterward — it means the PSF
is baked into the image-formation process itself, not an add-on that could in principle be skipped.
</details>

## 🧠 Questions & discussion

- If diffraction is unavoidable, why do real microscopes bother using high-NA objectives and
  immersion oil at all, rather than accepting a fixed, "good enough" resolution limit?
- Structured illumination and other super-resolution techniques are sometimes described as "beating
  the diffraction limit." Given that the OTF cutoff is a hard zero, what must such a technique
  actually be doing differently, rather than literally increasing the cutoff frequency of a single
  diffraction-limited exposure? (A later module on confocal, multiphoton, and super-resolution optics
  returns to this question directly.)
- This module treated the PSF as fixed and the same everywhere in the field of view. Is that always
  a safe assumption for a real microscope? What might make the PSF vary by position?

## 🚀 Bonus — going further

- Compute the Rayleigh resolution and OTF cutoff for a confocal microscope's *effective* PSF (the
  product of an excitation and a detection PSF) and compare it to the single-pass widefield PSF
  computed in this module.
- Add a small amount of defocus (a quadratic phase term across the pupil) to `simulate_psf` and watch
  how quickly the Airy rings degrade — this previews Module 05's treatment of aberrations.
- Look up the Nyquist-Shannon sampling criterion as applied to microscopy (commonly stated as
  "2-3 camera pixels per Rayleigh radius") and check whether Section 5's `pixel_um` choice satisfies
  it for the `na`/`wavelength_um` used there.

## 📝 Lab note

```text
Module: 04 — Diffraction, Resolution, and the Point-Spread Function
Date:
The NA and wavelength I would plug into simulate_psf for MY own imaging question:
One claim about "resolution" I would now double-check before trusting a vendor's spec sheet:
```

## 📚 References

1. Goodman, J.W. (2017). *Introduction to Fourier Optics* (4th ed.). W. H. Freeman.
2. Rayleigh, Lord (1879). Investigations in optics, with special reference to the spectroscope.
   *Philosophical Magazine*, 8(49), 261-274.
3. Abbe, E. (1873). Beitrage zur Theorie des Mikroskops und der mikroskopischen Wahrnehmung.
   *Archiv fur Mikroskopische Anatomie*, 9(1), 413-468.
4. Pawley, J.B. (Ed.) (2006). *Handbook of Biological Confocal Microscopy* (3rd ed.). Springer.
5. Murphy, D.B., Davidson, M.W. (2012). *Fundamentals of Light Microscopy and Electronic Imaging*
   (2nd ed.). Wiley-Blackwell. — a fuller treatment of diffraction theory and resolution criteria.

## ➡️ Next

Module 05 adds back everything this module's "perfect lens" assumption left out: real lenses are
imperfect, and those imperfections — aberrations — degrade the PSF in specific, physically
predictable ways, described by the same Zernike polynomials used here to define the pupil.